# Week 3 Task: Unsupervised Learning and Clustering Analysis

## Dataset: Mall Customers
This notebook performs data inspection, preprocessing, K-Means clustering, Elbow Method, Silhouette Analysis, visualization, cluster profiling, and Hierarchical Clustering.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage


## 1. Load the Dataset

In [ ]:
# Keep Mall_Customers.csv in the same folder as this notebook.
df = pd.read_csv("Mall_Customers.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)
df.head()


## 2. Understand the Dataset

In [ ]:
print("--- Dataset Information ---")
df.info()

print("\n--- First 5 Rows ---")
display(df.head())

print("\n--- Statistical Summary ---")
display(df.describe())

print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Duplicate Rows ---")
print(df.duplicated().sum())


## 3. Select Features for Clustering

CustomerID is only an identifier, so it is not useful for clustering. We use Age, Annual Income, and Spending Score.

In [ ]:
X = df[[
    "Age",
    "Annual Income (k$)",
    "Spending Score (1-100)"
]].copy()

print("Features selected:")
display(X.head())


## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    s=80
)
plt.title("Annual Income vs Spending Score")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.grid(True)
plt.show()


## 5. Standardize the Features

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("First 5 scaled rows:")
print(X_scaled[:5])


## 6. Elbow Method

The Elbow Method compares the within-cluster sum of squares (inertia) for different values of K. The point where the decrease starts slowing down is used as a practical choice for K.

In [ ]:
inertia = []
k_values = range(1, 11)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_values, inertia, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()


## 7. Silhouette Analysis

Silhouette Score measures how well each point fits its own cluster compared with other clusters. A higher score generally indicates better-separated clusters.

In [ ]:
silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

silhouette_results = pd.DataFrame({
    "K": list(k_range),
    "Silhouette Score": silhouette_scores
})
display(silhouette_results)

best_k_silhouette = int(silhouette_results.loc[
    silhouette_results["Silhouette Score"].idxmax(), "K"
])
print("Best K according to silhouette score:", best_k_silhouette)

plt.figure(figsize=(8, 5))
plt.plot(list(k_range), silhouette_scores, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different K Values")
plt.grid(True)
plt.show()


## 8. Apply K-Means Clustering

For this project, K=5 is used for the final customer segmentation, which is also a common segmentation choice for this Mall Customers dataset. If your Elbow/Silhouette analysis gives a different clearly justified value, you can change `final_k`.

In [ ]:
final_k = 5

kmeans_final = KMeans(
    n_clusters=final_k,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans_final.fit_predict(X_scaled)

df["Cluster"] = cluster_labels

print("K-Means clustering completed.")
print("Number of clusters:", final_k)
display(df.head())


## 9. Visualize the K-Means Clusters

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Cluster",
    palette="viridis",
    s=90
)

centers_original = scaler.inverse_transform(kmeans_final.cluster_centers_)
plt.scatter(
    centers_original[:, 1],
    centers_original[:, 2],
    marker="X",
    s=250,
    label="Centroids"
)

plt.title("Customer Segmentation using K-Means")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.legend(title="Cluster")
plt.grid(True)
plt.show()


## 10. Cluster Size

In [ ]:
cluster_counts = df["Cluster"].value_counts().sort_index()
print("Customers in each cluster:")
display(cluster_counts.to_frame("Customer Count"))

plt.figure(figsize=(8, 5))
cluster_counts.plot(kind="bar")
plt.xlabel("Cluster")
plt.ylabel("Number of Customers")
plt.title("Number of Customers in Each Cluster")
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.show()


## 11. Analyze the Characteristics of Each Cluster

In [ ]:
cluster_summary = df.groupby("Cluster")[[
    "Age",
    "Annual Income (k$)",
    "Spending Score (1-100)"
]].mean().round(2)

cluster_summary["Customer Count"] = cluster_counts

print("Average characteristics of each cluster:")
display(cluster_summary)


### Interpretation guide

- **High income + high spending:** potentially valuable/high-value customers.
- **High income + low spending:** customers with purchasing power but lower current engagement.
- **Low income + high spending:** customers who spend relatively actively despite lower income.
- **Low income + low spending:** lower-engagement segment.
- **Middle-income / middle-spending:** relatively average customers.

Use the actual `cluster_summary` values above to describe the clusters rather than assuming the cluster numbers themselves have a fixed meaning.

## 12. Hierarchical Clustering

In [ ]:
linked = linkage(X_scaled, method="ward")

plt.figure(figsize=(12, 6))
dendrogram(linked)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Customers")
plt.ylabel("Euclidean Distance")
plt.grid(True)
plt.show()


In [ ]:
hierarchical = AgglomerativeClustering(n_clusters=final_k, linkage="ward")
hierarchical_labels = hierarchical.fit_predict(X_scaled)

df["Hierarchical_Cluster"] = hierarchical_labels

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Hierarchical_Cluster",
    palette="plasma",
    s=90
)
plt.title("Hierarchical Clustering")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.legend(title="Cluster")
plt.grid(True)
plt.show()


## 13. Compare K-Means and Hierarchical Clustering

In [ ]:
kmeans_silhouette = silhouette_score(X_scaled, cluster_labels)
hierarchical_silhouette = silhouette_score(X_scaled, hierarchical_labels)

comparison = pd.DataFrame({
    "Algorithm": ["K-Means", "Hierarchical Clustering"],
    "Number of Clusters": [final_k, final_k],
    "Silhouette Score": [kmeans_silhouette, hierarchical_silhouette]
})

display(comparison.round(4))


## 14. Final Conclusion

The Mall Customers dataset was analyzed using unsupervised learning techniques. The customer features Age, Annual Income, and Spending Score were standardized before clustering. K-Means was used to segment customers into groups, and the Elbow Method and Silhouette Analysis were used to evaluate the choice of K. Cluster averages were then examined to understand the characteristics of each segment. Hierarchical Clustering and a dendrogram were also used as a second clustering approach for comparison.

The resulting customer segments can help identify groups with different income and spending behaviors. These segments may support targeted marketing, customer engagement, and personalized promotional strategies.

In [ ]:
# Optional: save the final clustered dataset
df.to_csv("Mall_Customers_Clustered.csv", index=False)
print("Saved: Mall_Customers_Clustered.csv")
